In [1]:
#%pip install torchvision

In [33]:
import torch
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [34]:
# images load => transform => dataset of all imgs
class ImageProcessor:
    def __init__(self, root_dir_path, transformations=None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        # Only load image files; folders and metadata files are not images.
        valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
        self.all_img_paths = [
            os.path.join(root_dir_path, img)
            for img in os.listdir(root_dir_path)
            if os.path.isfile(os.path.join(root_dir_path, img))
            and os.path.splitext(img)[1].lower() in valid_extensions
        ]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, idx):
        img_path = self.all_img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [35]:
root_dir_path = "./img_align_celeba"

transformations = transforms.Compose([
    transforms.CenterCrop(178),  # 178x218 => 178x178
    transforms.Resize(64), # 64x64
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) #[-1,1]
])

In [36]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"loaded {len(dataset)} images")

loaded 202599 images


In [37]:
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

### Generator Network

In [38]:
import torch.nn as nn 
import torch.optim as optim
import numpy as np

In [39]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, img_channels=3): 
        super(Generator, self).__init__()

        # fully connected (dense) layers
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256),  # 100 => 256
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 1024),
            nn.ReLU(),

            nn.Linear(1024, img_channels * 64 * 64),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 3, 64, 64)
        return img

### Discriminator Network

In [40]:
class Discriminator(nn.Module):
    def __init__(self, img_channels=3):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            nn.Flatten(),  # 4D tensor => 1D

            nn.Linear(img_channels *64 *64, 1024), 
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(256, 1)
        )

    def forward(self, z):
        return self.model(z)

In [41]:
import torch
# device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"device is {device}")

device is cpu


In [42]:
GAN_loss = nn.BCEWithLogitsLoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

generator = generator.to(device)
discriminator = discriminator.to(device)

### Training the GAN

In [43]:
def train(generator, discriminator, dataloader, checkpoint_path="gan_checkpoint.pth", resume=False):

    start_epoch = 0
    if resume:
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
        generator.load_state_dict(checkpoint["generator_state_dict"])
        discriminator.load_state_dict(checkpoint["discriminator_state_dict"])
        g_optimizer.load_state_dict(checkpoint["g_optimizer_state_dict"])
        d_optimizer.load_state_dict(checkpoint["d_optimizer_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        fixed_noise = checkpoint["fixed_noise"].to(device)
    else:
        fixed_noise = torch.randn(8, 100, device=device)

    epoch = start_epoch
    while True:
        for i, imgs in enumerate(dataloader):
            real_imgs = imgs.to(device)
            batch_size = real_imgs.size(0)

            # creates real imgs labels & fake imgs labels
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            # train the Discriminator
            d_optimizer.zero_grad()

            fake_imgs = generator(torch.randn(batch_size, 100).to(device))

            real_loss = GAN_loss(discriminator(real_imgs), real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()), fake_labels)

            d_loss = (real_loss + fake_loss)/2

            d_loss.backward()
            d_optimizer.step()

            # train the Generator
            for param in discriminator.parameters():
                param.requires_grad_(False)

            g_optimizer.zero_grad()

            g_loss = GAN_loss(discriminator(fake_imgs), real_labels)

            g_loss.backward()
            g_optimizer.step()

            for param in discriminator.parameters():
                param.requires_grad_(True)

            if i % 50 == 0:
                print(f"for epoch: {epoch+1}... batch: {i+1}... G-loss: {g_loss}... D-loss: {d_loss}")

        # Save training state after each completed epoch so training can be resumed.
        checkpoint_dir = os.path.dirname(checkpoint_path)
        if checkpoint_dir:
            os.makedirs(checkpoint_dir, exist_ok=True)
        torch.save({
            "epoch": epoch,
            "generator_state_dict": generator.state_dict(),
            "discriminator_state_dict": discriminator.state_dict(),
            "g_optimizer_state_dict": g_optimizer.state_dict(),
            "d_optimizer_state_dict": d_optimizer.state_dict(),
            "fixed_noise": fixed_noise.cpu(),
        }, checkpoint_path)
        save_generated_images(generator, epoch, device, fixed_noise)
        epoch += 1

In [44]:
import matplotlib.pyplot as plt
import torchvision

def save_generated_images(generator, epoch, device, fixed_noise):
    was_training = generator.training
    generator.eval()
    with torch.no_grad():
        generated_imgs = generator(fixed_noise).cpu()
    if was_training:
        generator.train()

    grid = torchvision.utils.make_grid(generated_imgs, nrow=4, normalize=True)

    plt.imshow(np.transpose(grid, (1,2,0)))
    plt.title(f"epoch {epoch+1}")
    plt.axis("off")
    plt.show()

In [45]:
train(generator, discriminator, dataloader)

for epoch: 1... batch: 1... G-loss: 0.6979495882987976... D-loss: 0.6939399838447571
for epoch: 1... batch: 51... G-loss: 1.0453593730926514... D-loss: 0.2701133191585541
for epoch: 1... batch: 101... G-loss: 1.4781320095062256... D-loss: 0.17327114939689636
for epoch: 1... batch: 151... G-loss: 0.7102254629135132... D-loss: 0.38314735889434814
for epoch: 1... batch: 201... G-loss: 2.2492103576660156... D-loss: 0.08820723742246628
for epoch: 1... batch: 251... G-loss: 1.7347809076309204... D-loss: 0.10703982412815094
for epoch: 1... batch: 301... G-loss: 3.493910789489746... D-loss: 0.23044581711292267
for epoch: 1... batch: 351... G-loss: 2.3698208332061768... D-loss: 0.06909232586622238
for epoch: 1... batch: 401... G-loss: 2.8455522060394287... D-loss: 0.1551288664340973
for epoch: 1... batch: 451... G-loss: 4.402851581573486... D-loss: 0.13295966386795044
for epoch: 1... batch: 501... G-loss: 4.0845489501953125... D-loss: 0.24617305397987366
for epoch: 1... batch: 551... G-loss: 0.

UnidentifiedImageError: cannot identify image file './img_align_celeba\\desktop.ini'